In [ ]:
# ══════════════════════════════════════════════════════════
# DESCARGA DE DATOS — Solo corre en Google Colab
# En local los archivos ya están en la carpeta data/
# ══════════════════════════════════════════════════════════
import os
import requests

def descargar(url, destino, nombre):
    headers = {'User-Agent': 'Mozilla/5.0'}
    r = requests.get(url, headers=headers)
    if r.status_code == 200:
        with open(destino, 'wb') as f:
            f.write(r.content)
        print(f'✓ {nombre} descargado')
    else:
        print(f'✗ Error {r.status_code} al descargar {nombre}')

if not os.path.exists('data'):
    os.makedirs('data')
    print('Descargando datos...\n')

    # IPC — INDEC vía datos.gob.ar
    descargar(
        'https://infra.datos.gob.ar/catalog/sspm/dataset/145/distribution/145.3/download/indice-precios-al-consumidor-nivel-general-base-diciembre-2016-mensual.csv',
        'data/ipc.csv', 'IPC'
    )

    # Tipo de cambio BNA — BCRA vía apis.datos.gob.ar
    descargar(
        'https://apis.datos.gob.ar/series/api/series/?ids=168.1_T_CAMBIOR_D_0_0_26&format=csv&start_date=2016-01-01&limit=5000&collapse=month&collapse_aggregation=avg',
        'data/tipo_cambio.csv', 'Tipo de cambio'
    )

    # SMVM — viene del repositorio
    descargar(
        'https://raw.githubusercontent.com/MartinCarossino/indice-tanque-lleno/main/data/smvm_historico.csv',
        'data/smvm_historico.csv', 'SMVM'
    )

    # Nafta — viene del repositorio
    descargar(
        'https://raw.githubusercontent.com/MartinCarossino/indice-tanque-lleno/main/data/nafta.csv',
        'data/nafta.csv', 'Nafta'
    )

    print('\n✓ Todos los datos listos.')
else:
    print('✓ Carpeta data/ ya existe — usando archivos locales.')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("muted")

print("✓ Librerías cargadas correctamente")
print(f"  pandas  {pd.__version__}")
print(f"  numpy   {np.__version__}")

In [ ]:
# ══════════════════════════════════════════════════════════
# SECCIÓN 01 — Carga y exploración inicial
# ══════════════════════════════════════════════════════════

nafta_raw = pd.read_csv('data/nafta.csv', low_memory=False)
ipc_raw   = pd.read_csv('data/ipc.csv')
smvm_raw  = pd.read_csv('data/smvm_historico.csv')
tc_raw    = pd.read_csv('data/tipo_cambio.csv')

print(f"nafta : {nafta_raw.shape[0]:>7,} filas × {nafta_raw.shape[1]} columnas")
print(f"ipc   : {ipc_raw.shape[0]:>7,} filas × {ipc_raw.shape[1]} columnas")
print(f"smvm  : {smvm_raw.shape[0]:>7,} filas × {smvm_raw.shape[1]} columnas")
print(f"tc    : {tc_raw.shape[0]:>7,} filas × {tc_raw.shape[1]} columnas")

In [ ]:
# ── Vista rápida de los cuatro datasets ───────────────────
print("=== NAFTA (primeras 3 filas) ===")
display(nafta_raw[['indice_tiempo','producto','precio','provincia','empresabandera']].head(3))

print("\n=== IPC (primeras 3 filas) ===")
display(ipc_raw[['indice_tiempo','ipc_ng_nacional']].head(3))

print("\n=== SMVM (primeras 3 filas) ===")
display(smvm_raw.head(3))

print("\n=== TIPO DE CAMBIO (primeras 3 filas) ===")
display(tc_raw.head(3))

In [ ]:
# ── Nulos y rangos de fechas ───────────────────────────────
datasets = {
    'nafta' : nafta_raw,
    'ipc'   : ipc_raw,
    'smvm'  : smvm_raw,
    'tc'    : tc_raw
}

for nombre, df in datasets.items():
    nulos = df.isnull().sum().sum()
    fecha_col = df.columns[0]
    print(f"{nombre:6} | filas: {len(df):>6,} | nulos totales: {nulos:>5} | rango: {df[fecha_col].min()} → {df[fecha_col].max()}")

In [ ]:
# ══════════════════════════════════════════════════════════
# SECCIÓN 02 — Limpieza y preparación
# ══════════════════════════════════════════════════════════

# ── Nafta: filtrar súper y promediar por mes ───────────────
# Solo nafta súper (la más representativa del mercado)
# Como hay un precio por estación, promediamos por mes
# Decisión analítica: lo documentamos en el notebook

nafta = nafta_raw[
    nafta_raw['producto'] == 'Nafta (súper) entre 92 y 95 Ron'
].copy()

# Parsear fecha y agrupar por mes
nafta['fecha'] = pd.to_datetime(nafta['indice_tiempo'], format='%Y-%m')

nafta = (
    nafta
    .groupby('fecha')['precio']
    .mean()
    .reset_index()
    .rename(columns={'precio': 'precio_nafta'})
)

print(f"Filas después de filtrar: {len(nafta)}")
print(f"Rango: {nafta['fecha'].min().strftime('%b %Y')} → {nafta['fecha'].max().strftime('%b %Y')}")
display(nafta.head(3))

In [ ]:
# ── IPC: quedarnos solo con fecha e índice nacional ────────
ipc = ipc_raw[['indice_tiempo', 'ipc_ng_nacional']].copy()
ipc['fecha'] = pd.to_datetime(ipc['indice_tiempo'])
ipc = ipc.drop(columns='indice_tiempo')

print(f"Filas: {len(ipc)}")
print(f"Rango: {ipc['fecha'].min().strftime('%b %Y')} → {ipc['fecha'].max().strftime('%b %Y')}")
display(ipc.head(3))

In [ ]:
# ── SMVM: calcular salario diario ─────────────────────────
# Dividimos por 30 para obtener un día de trabajo mínimo
# Esa es la unidad del índice del tanque lleno

smvm = smvm_raw.copy()
smvm['fecha'] = pd.to_datetime(smvm['indice_tiempo'])
smvm['salario_diario'] = smvm['smvm_mensual'] / 30
smvm = smvm.drop(columns='indice_tiempo')

print(f"Filas: {len(smvm)}")
print(f"Rango: {smvm['fecha'].min().strftime('%b %Y')} → {smvm['fecha'].max().strftime('%b %Y')}")
display(smvm.head(3))

In [ ]:
# ── Tipo de cambio: parsear fecha ─────────────────────────
tc = tc_raw.copy()
tc['fecha'] = pd.to_datetime(tc['indice_tiempo'])
tc = tc.rename(columns={'tipo_cambio_bna_vendedor': 'tipo_cambio'})
tc = tc.drop(columns='indice_tiempo')

print(f"Filas: {len(tc)}")
print(f"Rango: {tc['fecha'].min().strftime('%b %Y')} → {tc['fecha'].max().strftime('%b %Y')}")
display(tc.head(3))

In [ ]:
# ── Merge de los cuatro datasets ──────────────────────────
# inner join: solo meses donde TODOS tienen dato
df = (
    nafta
    .merge(ipc,  on='fecha', how='inner')
    .merge(smvm, on='fecha', how='inner')
    .merge(tc,   on='fecha', how='inner')
)

print(f"Dataset final: {df.shape}")
print(f"Rango: {df['fecha'].min().strftime('%b %Y')} → {df['fecha'].max().strftime('%b %Y')}")
print(f"\nNulos por columna:")
print(df.isnull().sum())
display(df.head())

In [ ]:
# ══════════════════════════════════════════════════════════
# SECCIÓN 03 — Evolución nominal del precio
# ══════════════════════════════════════════════════════════

fig, ax = plt.subplots()

ax.plot(df['fecha'], df['precio_nafta'], color='#E8A020', linewidth=2)

# Anotaciones en los saltos más importantes
hitos = {
    '2018-09-01': ('Crisis cambiaria\nsep 2018',  'bottom'),
    '2020-04-01': ('Pandemia\nabr 2020',          'top'),
    '2023-12-01': ('Devaluación\ndic 2023',       'bottom'),
}

for fecha_str, (etiqueta, posicion) in hitos.items():
    fecha = pd.Timestamp(fecha_str)
    valor = df.loc[df['fecha'] == fecha, 'precio_nafta']
    if not valor.empty:
        offset = 60 if posicion == 'top' else -120
        ax.annotate(
            etiqueta,
            xy=(fecha, valor.values[0]),
            xytext=(0, offset),
            textcoords='offset points',
            ha='center', fontsize=8, color='#aaaaaa',
            arrowprops=dict(arrowstyle='->', color='#666666', lw=0.8)
        )

ax.set_title('Precio de la nafta súper en Argentina (2017–2026)', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('Precio por litro ($ARS)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

# Estadística rápida
precio_min = df['precio_nafta'].min()
precio_max = df['precio_nafta'].max()
variacion  = (precio_max / precio_min - 1) * 100
print(f"Precio mínimo : ${precio_min:,.2f}  ({df.loc[df['precio_nafta'].idxmin(), 'fecha'].strftime('%b %Y')})")
print(f"Precio máximo : ${precio_max:,.2f}  ({df.loc[df['precio_nafta'].idxmax(), 'fecha'].strftime('%b %Y')})")
print(f"Variación total: +{variacion:,.0f}%")

In [ ]:
fig, ax = plt.subplots()

ax.plot(df['fecha'], df['precio_nafta'], color='#E8A020', linewidth=2)

hitos = {
    '2018-09-01': ('Crisis cambiaria\nsep 2018',  'bottom'),
    '2020-06-01': ('Congelamiento\npandemia',      'top'),
    '2023-12-01': ('Devaluación\ndic 2023',        'bottom'),
}

for fecha_str, (etiqueta, posicion) in hitos.items():
    fecha = pd.Timestamp(fecha_str)
    valor = df.loc[df['fecha'] == fecha, 'precio_nafta']
    if not valor.empty:
        offset = 60 if posicion == 'top' else -120
        ax.annotate(
            etiqueta,
            xy=(fecha, valor.values[0]),
            xytext=(0, offset),
            textcoords='offset points',
            ha='center', fontsize=8, color='#aaaaaa',
            arrowprops=dict(arrowstyle='->', color='#666666', lw=0.8)
        )

ax.set_title('Precio de la nafta súper en Argentina (2017–2026)', fontsize=13, pad=12)
ax.set_xlabel('')
ax.set_ylabel('Precio por litro ($ARS)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

precio_min = df['precio_nafta'].min()
precio_max = df['precio_nafta'].max()
variacion  = (precio_max / precio_min - 1) * 100
print(f"Precio mínimo : ${precio_min:,.2f}  ({df.loc[df['precio_nafta'].idxmin(), 'fecha'].strftime('%b %Y')})")
print(f"Precio máximo : ${precio_max:,.2f}  ({df.loc[df['precio_nafta'].idxmax(), 'fecha'].strftime('%b %Y')})")
print(f"Variación total: +{variacion:,.0f}%")

In [ ]:
# ══════════════════════════════════════════════════════════
# SECCIÓN 04 — Precio real deflactado
# ══════════════════════════════════════════════════════════

# Base: enero 2017 (primer mes del dataset)
# Precio real = precio_nominal / (IPC / IPC_base)
# Responde: ¿cuánto valdría la nafta de hoy en pesos de ene-2017?

ipc_base = df.loc[df['fecha'] == '2017-01-01', 'ipc_ng_nacional'].values[0]
df['precio_nafta_real'] = df['precio_nafta'] / (df['ipc_ng_nacional'] / ipc_base)

# ── Gráfico: nominal vs. real ──────────────────────────────
fig, ax = plt.subplots()

ax.plot(df['fecha'], df['precio_nafta'],      color='#E8A020', linewidth=2, label='Precio nominal')
ax.plot(df['fecha'], df['precio_nafta_real'], color='#5B9BD5', linewidth=2, linestyle='--', label='Precio real (pesos ene-2017)')

ax.set_title('Nafta súper: precio nominal vs. precio real (2017–2026)', fontsize=13, pad=12)
ax.set_ylabel('Precio por litro ($ARS)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()

plt.tight_layout()
plt.show()

# ── ¿La nafta subió más o menos que la inflación? ─────────
ultimo  = df.iloc[-1]
primero = df.iloc[0]

variacion_nominal = (ultimo['precio_nafta']      / primero['precio_nafta']      - 1) * 100
variacion_ipc     = (ultimo['ipc_ng_nacional']   / primero['ipc_ng_nacional']   - 1) * 100
variacion_real    = (ultimo['precio_nafta_real'] / primero['precio_nafta_real'] - 1) * 100

print(f"Variación nominal de la nafta : +{variacion_nominal:,.0f}%")
print(f"Variación del IPC (inflación) : +{variacion_ipc:,.0f}%")
print(f"Variación real de la nafta    : {variacion_real:+,.1f}%")
print()
if variacion_real > 0:
    print(f"➜ La nafta subió MÁS que la inflación: +{variacion_real:.1f}% en términos reales")
else:
    print(f"➜ La nafta subió MENOS que la inflación: {variacion_real:.1f}% en términos reales")

In [ ]:
# ══════════════════════════════════════════════════════════
# SECCIÓN 05 — El índice del tanque lleno
# ══════════════════════════════════════════════════════════

# ── Cálculo del índice ────────────────────────────────────
# ¿Cuántos litros de nafta súper podés comprar
# con un día de trabajo al salario mínimo?

df['litros_por_dia'] = df['salario_diario'] / df['precio_nafta']

# ── Gráfico ───────────────────────────────────────────────
fig, ax = plt.subplots()

ax.plot(df['fecha'], df['litros_por_dia'], color='#2ECC71', linewidth=2)
ax.axhline(y=df['litros_por_dia'].mean(), color='#888888', linewidth=1,
           linestyle=':', label=f"Promedio: {df['litros_por_dia'].mean():.1f} L")

ax.fill_between(df['fecha'], df['litros_por_dia'],
                df['litros_por_dia'].mean(),
                where=df['litros_por_dia'] >= df['litros_por_dia'].mean(),
                alpha=0.15, color='#2ECC71', label='Por encima del promedio')
ax.fill_between(df['fecha'], df['litros_por_dia'],
                df['litros_por_dia'].mean(),
                where=df['litros_por_dia'] < df['litros_por_dia'].mean(),
                alpha=0.15, color='#E74C3C', label='Por debajo del promedio')

ax.set_title('Índice del tanque lleno — litros por día de salario mínimo (2017–2026)', fontsize=12, pad=12)
ax.set_ylabel('Litros de nafta súper')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# ── Estadística del índice ────────────────────────────────
idx_max = df['litros_por_dia'].idxmax()
idx_min = df['litros_por_dia'].idxmin()

print(f"Mejor momento  : {df.loc[idx_max, 'fecha'].strftime('%b %Y')} → {df.loc[idx_max, 'litros_por_dia']:.1f} litros/día")
print(f"Peor momento   : {df.loc[idx_min, 'fecha'].strftime('%b %Y')} → {df.loc[idx_min, 'litros_por_dia']:.1f} litros/día")
print(f"Promedio serie : {df['litros_por_dia'].mean():.1f} litros/día")
print(f"Valor actual   : {df['litros_por_dia'].iloc[-1]:.1f} litros/día")
print()
variacion_indice = (df['litros_por_dia'].iloc[-1] / df['litros_por_dia'].iloc[0] - 1) * 100
print(f"Variación 2017→2026: {variacion_indice:+.1f}%")

In [ ]:
# ══════════════════════════════════════════════════════════
# SECCIÓN 06 — Precio en dólares
# ══════════════════════════════════════════════════════════

# ── Precio del litro en USD ───────────────────────────────
df['precio_nafta_usd'] = df['precio_nafta'] / df['tipo_cambio']

# ── Gráfico ───────────────────────────────────────────────
fig, ax = plt.subplots()

ax.plot(df['fecha'], df['precio_nafta_usd'], color='#85C1E9', linewidth=2)
ax.axhline(y=df['precio_nafta_usd'].mean(), color='#888888', linewidth=1,
           linestyle=':', label=f"Promedio: U$S {df['precio_nafta_usd'].mean():.2f}")

ax.set_title('Precio de la nafta súper en dólares (2017–2026)', fontsize=13, pad=12)
ax.set_ylabel('USD por litro')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'U$S {x:.2f}'))
ax.legend()

plt.tight_layout()
plt.show()

# ── Estadística ───────────────────────────────────────────
idx_max = df['precio_nafta_usd'].idxmax()
idx_min = df['precio_nafta_usd'].idxmin()

print(f"Precio máximo en USD : U$S {df.loc[idx_max, 'precio_nafta_usd']:.3f}  ({df.loc[idx_max, 'fecha'].strftime('%b %Y')})")
print(f"Precio mínimo en USD : U$S {df.loc[idx_min, 'precio_nafta_usd']:.3f}  ({df.loc[idx_min, 'fecha'].strftime('%b %Y')})")
print(f"Precio promedio      : U$S {df['precio_nafta_usd'].mean():.3f}")
print(f"Precio actual        : U$S {df['precio_nafta_usd'].iloc[-1]:.3f}")

# Conclusiones — Índice del Tanque Lleno

## Hallazgos principales

**1. El precio nominal se multiplicó por 83**
De $20.71 en enero 2017 a $1.724 en enero 2026. Un aumento de +8.227% en pesos corrientes.

**2. En términos reales, la nafta es más barata que en 2017**
La inflación acumulada (+10.150%) superó al aumento de la nafta (+8.227%).
En pesos constantes de 2017, el precio real cayó un **18.8%**.

**3. El poder adquisitivo para comprar combustible cayó un 18.5%**
En 2017 un trabajador al salario mínimo compraba **12.9 litros** con un día de trabajo.
En 2026 compra **10.6 litros**. Perdió casi 2.5 litros de capacidad de compra.

**4. El peor momento fue julio 2022: 5.2 litros/día**
En plena crisis macroeconómica, el salario mínimo perdió poder adquisitivo frente al combustible.

**5. En dólares, la nafta argentina es históricamente estable**
Promedio de U$S 1.07 el litro durante 9 años. Hoy U$S 1.17, levemente por encima del promedio.

## Limitaciones del análisis
- Se usó el precio **promedio nacional** de nafta súper. Los precios varían por provincia y empresa.
- El SMVM no representa el salario real de la mayoría de los trabajadores, sino el piso legal.
- El tipo de cambio usado es el oficial BNA. El dólar blue daría resultados distintos.
- Los datos de nafta del archivo vigente arrancan en junio 2016 pero el IPC base dic-2016
  limita el análisis efectivo a partir de enero 2017.